# Employee Attrition Prediction - IBM HR Analytics
**CSE570 - Machine Learning with Python | Skill Based Assignment**

**Problem:** Predict whether an employee will leave (`Attrition` = Yes/No) - binary classification.
**Dataset:** IBM HR Analytics Employee Attrition & Performance (Kaggle, `WA_Fn-UseC_-HR-Employee-Attrition.csv`), 1470 rows x 35 columns.
**Success criteria:** ROC-AUC >= 0.80 and recall on the 'Left' class >= 0.60 on the held-out test set (the cost of missing a leaver is higher than a false alarm).

**Workflow:** load -> clean -> EDA + statistical tests -> split -> leak-free pipelines -> baseline + 6 models with GridSearchCV -> compare -> final model + interpretation -> limitations.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report, roc_curve)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
os.makedirs('outputs', exist_ok=True)

def save(name):
    plt.savefig(f'outputs/{name}.png', dpi=150, bbox_inches='tight')

## 1. Load data

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()
print()
print(df.describe().T.round(2))

## 2. Data cleaning
Steps: missing values -> duplicates -> constant/ID columns -> datatype review -> outlier inspection.

In [ ]:
# Missing values and duplicates
print('Total missing values:', df.isnull().sum().sum())
print('Duplicate rows     :', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# Constant columns carry no information; EmployeeNumber is just an ID
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print('Constant columns dropped:', constant_cols, '+ EmployeeNumber')
df = df.drop(columns=constant_cols + ['EmployeeNumber'])

# Datatype review: object columns are nominal categories. Ordinal survey scores
# (1-4 / 1-5 ratings) are kept numeric so their order is preserved.
df['AttritionFlag'] = (df['Attrition'] == 'Yes').astype(int)   # Yes=1, No=0 (used in EDA)
cat_cols_all = df.select_dtypes(include='object').columns.drop('Attrition').tolist()
print('Categorical columns:', cat_cols_all)
print('Shape after cleaning:', df.shape)

In [ ]:
# Outlier INSPECTION (IQR rule) on genuinely continuous features only
cont_features = ['Age', 'MonthlyIncome', 'DistanceFromHome', 'TotalWorkingYears',
                 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']
rows = []
for c in cont_features:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    n_out = ((df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)).sum()
    rows.append((c, n_out, round(100*n_out/len(df), 1), round(df[c].skew(), 2)))
outlier_df = pd.DataFrame(rows, columns=['Feature', 'Outliers', 'Outlier_%', 'Skew']).sort_values('Outliers', ascending=False)
print(outlier_df.to_string(index=False))

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, c in zip(axes.ravel(), cont_features):
    sns.boxplot(y=df[c], ax=ax, color='lightblue'); ax.set_title(c)
plt.tight_layout(); save('outliers_boxplots'); plt.show()

**Outlier decision:** the flagged values (high income, long tenure) are genuine, plausible employee records, not errors, so they are **kept**. Tree-based models are insensitive to them, and the distance/linear models use `StandardScaler`, with regularisation limiting their influence. Scaling and encoding are done *inside the pipelines* (Section 5) so no statistics leak from the test set.

## 3. Exploratory Data Analysis

### 3.1 Univariate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='Attrition', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Attrition count')
df['Attrition'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1], colors=['lightgreen', 'lightcoral'])
axes[1].set_ylabel(''); axes[1].set_title('Attrition distribution')
plt.tight_layout(); save('target_distribution'); plt.show()
print(f"Attrition rate: {df['AttritionFlag'].mean()*100:.2f}%  (imbalanced -> use stratification, class weights, F1/recall/AUC, not accuracy)")

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, c in zip(axes.ravel(), cont_features):
    sns.histplot(df[c], kde=True, ax=ax, color='skyblue'); ax.set_title(f'{c} (skew={df[c].skew():.2f})')
plt.tight_layout(); save('univariate_hist'); plt.show()

### 3.2 Bivariate

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
sns.boxplot(x='Attrition', y='Age', data=df, ax=axes[0,0], palette='Set1'); axes[0,0].set_title('Age by Attrition')
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df, ax=axes[0,1], palette='Set1'); axes[0,1].set_title('Monthly income by Attrition')
sns.boxplot(x='Attrition', y='YearsAtCompany', data=df, ax=axes[0,2], palette='Set1'); axes[0,2].set_title('Years at company by Attrition')
for ax, c in zip(axes[1], ['OverTime', 'Department', 'MaritalStatus']):
    (df.groupby(c)['AttritionFlag'].mean()*100).plot(kind='bar', ax=ax, color='coral')
    ax.set_title(f'Attrition rate (%) by {c}'); ax.set_ylabel('%'); ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); save('bivariate'); plt.show()

plt.figure(figsize=(12, 5))
(df.groupby('JobRole')['AttritionFlag'].mean()*100).sort_values(ascending=False).plot(kind='bar', color='steelblue')
plt.title('Attrition rate (%) by Job Role'); plt.ylabel('%'); plt.xticks(rotation=45, ha='right')
plt.tight_layout(); save('attrition_by_jobrole'); plt.show()

### 3.3 Multivariate

In [ ]:
num_for_corr = df.select_dtypes(include=np.number).columns
corr = df[num_for_corr].corr()
print('Top correlations with Attrition:')
print(corr['AttritionFlag'].drop('AttritionFlag').sort_values(key=abs, ascending=False).head(10).round(3))

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation heatmap (numeric features + target)')
plt.tight_layout(); save('correlation_heatmap'); plt.show()

# Interaction view: attrition rate by OverTime x JobSatisfaction and OverTime x JobLevel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, c in zip(axes, ['JobSatisfaction', 'JobLevel']):
    sns.heatmap(df.pivot_table(index=c, columns='OverTime', values='AttritionFlag', aggfunc='mean')*100,
                annot=True, fmt='.1f', cmap='Reds', ax=ax)
    ax.set_title(f'Attrition rate (%): {c} x OverTime')
plt.tight_layout(); save('multivariate_pivots'); plt.show()

### 3.4 Statistical tests and confidence intervals

In [ ]:
def wilson_ci(k, n, z=1.96):
    p = k/n; d = 1 + z**2/n; c = p + z**2/(2*n)
    a = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))
    return (c-a)/d, (c+a)/d

# 95% CI for attrition rate overall and by OverTime
lo, hi = wilson_ci(df.AttritionFlag.sum(), len(df))
print(f'Overall attrition rate 95% CI: [{lo*100:.1f}%, {hi*100:.1f}%]')
for g, sub in df.groupby('OverTime'):
    lo, hi = wilson_ci(sub.AttritionFlag.sum(), len(sub))
    print(f'  OverTime={g}: {sub.AttritionFlag.mean()*100:.1f}%  95% CI [{lo*100:.1f}%, {hi*100:.1f}%]')

# Chi-square test of independence + Cramer's V for every categorical feature
res = []
for c in cat_cols_all:
    ct = pd.crosstab(df[c], df['Attrition'])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    res.append((c, round(chi2, 2), p, round(v, 3)))
chi_df = pd.DataFrame(res, columns=['Feature', 'Chi2', 'p_value', 'Cramers_V']).sort_values('p_value')
chi_df['Significant(0.05)'] = chi_df['p_value'] < 0.05
print('\nChi-square tests (categorical vs Attrition):')
print(chi_df.to_string(index=False))

# Numeric features: Mann-Whitney U (non-parametric, many features are skewed)
res = []
for c in cont_features:
    a = df.loc[df.Attrition == 'Yes', c]; b = df.loc[df.Attrition == 'No', c]
    u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    res.append((c, round(a.median(), 1), round(b.median(), 1), p))
mw_df = pd.DataFrame(res, columns=['Feature', 'Median_Left', 'Median_Stayed', 'p_value']).sort_values('p_value')
print('\nMann-Whitney U tests (Left vs Stayed):')
print(mw_df.to_string(index=False))

**Interpretation (fill in from your output):** write 4-5 sentences here - e.g. which features show a statistically significant association with attrition (p < 0.05), how strong the association is (Cramer's V / median gap), and what this means for HR. Keep it tied to the numbers printed above.

## 4. Train / test split
Stratified 80/20 split **before any fitting**. Hyper-parameters are tuned with 5-fold stratified CV on the training set only; the test set is used once at the end.

In [ ]:
X = df.drop(columns=['Attrition', 'AttritionFlag'])
y = df['AttritionFlag']
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Attrition rate - train {y_train.mean()*100:.2f}%, test {y_test.mean()*100:.2f}%')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 5. Models, hyper-parameter tuning, cross-validation
All preprocessing lives inside a `Pipeline` so scaling/encoding is fitted on the training folds only (**no data leakage**). Distance/linear/margin models get scaled numeric features; tree models get raw numerics. Class imbalance is handled with `class_weight`.

In [ ]:
def make_prep(scale):
    return ColumnTransformer(
        [('num', StandardScaler() if scale else 'passthrough', num_cols),
         ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)],
        sparse_threshold=0)

models = {
    'Baseline (Dummy)': (Pipeline([('prep', make_prep(True)), ('clf', DummyClassifier(strategy='prior'))]), {}),
    'Logistic Regression': (Pipeline([('prep', make_prep(True)), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))]),
                            {'clf__C': [0.01, 0.1, 1, 10]}),
    'KNN': (Pipeline([('prep', make_prep(True)), ('clf', KNeighborsClassifier())]),
            {'clf__n_neighbors': [3, 5, 9, 15, 21], 'clf__weights': ['uniform', 'distance']}),
    'Naive Bayes': (Pipeline([('prep', make_prep(True)), ('clf', GaussianNB())]),
                    {'clf__var_smoothing': np.logspace(-9, -1, 5)}),
    'SVM (RBF)': (Pipeline([('prep', make_prep(True)), ('clf', SVC(kernel='rbf', class_weight='balanced', random_state=RANDOM_STATE))]),
                  {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 0.01, 0.001]}),
    'Decision Tree': (Pipeline([('prep', make_prep(False)), ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE))]),
                      {'clf__max_depth': [3, 5, 7], 'clf__min_samples_leaf': [5, 10, 20]}),
    'Random Forest': (Pipeline([('prep', make_prep(False)), ('clf', RandomForestClassifier(class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1))]),
                      {'clf__n_estimators': [200, 400], 'clf__max_depth': [5, 10, None], 'clf__min_samples_leaf': [1, 3, 5]}),
}

def get_scores(est, X_):
    # probability of 'Left'; SVC (probability=False) falls back to decision_function
    return est.predict_proba(X_)[:, 1] if hasattr(est, 'predict_proba') else est.decision_function(X_)

scoring = {'f1': 'f1', 'roc_auc': 'roc_auc', 'recall': 'recall', 'precision': 'precision'}
fitted, rows = {}, []
for name, (pipe, grid) in models.items():
    gs = GridSearchCV(pipe, grid if grid else {'clf__strategy': ['prior']}, cv=cv,
                      scoring=scoring, refit='f1', n_jobs=-1)
    gs.fit(X_train, y_train)
    best = gs.best_estimator_; fitted[name] = best
    i = gs.best_index_
    pred = best.predict(X_test); score = get_scores(best, X_test)
    rows.append({'Model': name,
                 'Best params': {k.replace('clf__', ''): v for k, v in gs.best_params_.items()},
                 'CV_F1': gs.cv_results_['mean_test_f1'][i], 'CV_AUC': gs.cv_results_['mean_test_roc_auc'][i],
                 'Test_Accuracy': accuracy_score(y_test, pred), 'Test_Precision': precision_score(y_test, pred, zero_division=0),
                 'Test_Recall': recall_score(y_test, pred), 'Test_F1': f1_score(y_test, pred),
                 'Test_ROC_AUC': roc_auc_score(y_test, score)})
    print(f'done: {name}')

results = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 80, 'display.width', 250)
print(results.drop(columns='Best params').round(4).to_string(index=False))
print('\nBest hyper-parameters:')
for r in rows: print(f"  {r['Model']}: {r['Best params']}")
results.drop(columns='Best params').round(4).to_csv('outputs/model_comparison.csv', index=False)

## 6. Model comparison and final model selection
The final model is chosen by **cross-validated F1 on the training data** (not by the test set), then confirmed on the untouched test set.

In [ ]:
cand = results[results.Model != 'Baseline (Dummy)']
final_name = cand.loc[cand['CV_F1'].idxmax(), 'Model']
final_model = fitted[final_name]
print('Final model (highest CV F1):', final_name)

plot_df = results.set_index('Model')[['Test_Accuracy', 'Test_Precision', 'Test_Recall', 'Test_F1', 'Test_ROC_AUC']]
plot_df.plot(kind='bar', figsize=(14, 6), colormap='viridis')
plt.title('Test-set performance of all models'); plt.ylabel('Score'); plt.xticks(rotation=40, ha='right')
plt.legend(loc='lower right'); plt.tight_layout(); save('model_comparison'); plt.show()

# ROC curves
plt.figure(figsize=(9, 7))
for name, est in fitted.items():
    s = get_scores(est, X_test); fpr, tpr, _ = roc_curve(y_test, s)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc_score(y_test, s):.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=.5)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC curves (test set)')
plt.legend(loc='lower right'); plt.tight_layout(); save('roc_curves'); plt.show()

# Confusion matrices
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, (name, est) in zip(axes.ravel(), fitted.items()):
    sns.heatmap(confusion_matrix(y_test, est.predict(X_test)), annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'])
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
for ax in axes.ravel()[len(fitted):]: ax.axis('off')
plt.tight_layout(); save('confusion_matrices'); plt.show()

print(f'\nClassification report - {final_name} (test set)')
print(classification_report(y_test, final_model.predict(X_test), target_names=['Stayed', 'Left']))

## 7. Interpretation: what drives attrition?
Permutation importance is computed on the **test set through the full pipeline**, so it is reported per original feature (e.g. `JobRole`, not 6 dummy columns) and works for any model type.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring='roc_auc',
                              n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({'Feature': X_test.columns, 'Importance': perm.importances_mean, 'Std': perm.importances_std}) \
        .sort_values('Importance', ascending=False)
top = imp.head(15)
plt.figure(figsize=(10, 7))
sns.barplot(x='Importance', y='Feature', data=top, palette='viridis')
plt.title(f'Top 15 features - permutation importance ({final_name})'); plt.xlabel('Drop in ROC-AUC when shuffled')
plt.tight_layout(); save('feature_importance'); plt.show()
print(top.head(10).round(4).to_string(index=False))

## 8. Conclusion (numbers are printed from the actual results)

In [ ]:
final_row = results[results.Model == final_name].iloc[0]
base_row = results[results.Model == 'Baseline (Dummy)'].iloc[0]
print('PROJECT SUMMARY'); print('=' * 60)
print(f"Dataset: IBM HR Analytics, {df.shape[0]} records, attrition rate {df['AttritionFlag'].mean()*100:.1f}%")
print(f"Final model: {final_name}")
print(f"  Test ROC-AUC {final_row.Test_ROC_AUC:.3f} | Recall {final_row.Test_Recall:.3f} | Precision {final_row.Test_Precision:.3f} | F1 {final_row.Test_F1:.3f}")
print(f"  Baseline ROC-AUC {base_row.Test_ROC_AUC:.3f} | Baseline recall {base_row.Test_Recall:.3f}")
print('Top predictors:', ', '.join(imp.Feature.head(5)))
met = final_row.Test_ROC_AUC >= 0.80 and final_row.Test_Recall >= 0.60
print('Success criteria (AUC>=0.80 and recall>=0.60) met:', met)

### Limitations (edit to match your results)
- Single, publicly available **synthetic-style IBM dataset** (1,470 employees); results may not generalise to other companies.
- **Class imbalance** (~16% leavers): handled with class weights and stratified CV, but precision for the 'Left' class remains modest.
- Small test set (~294 rows, ~47 leavers) -> metrics have wide confidence intervals.
- Associations are **not causal** (e.g. overtime correlating with attrition does not prove overtime causes it).
- No time dimension: a snapshot, so we cannot model *when* employees leave.

### Future work
Threshold tuning for recall, SMOTE inside the CV pipeline, gradient boosting (XGBoost/LightGBM), SHAP explanations, survival analysis.

### References
- IBM HR Analytics Employee Attrition & Performance, Kaggle (pavansubhasht).
- Pedregosa et al., *Scikit-learn: Machine Learning in Python*, JMLR 12, 2011.
- McKinney (pandas), Hunter (matplotlib), Waskom (seaborn), Virtanen et al. (SciPy).